In [1]:
import requests
from langchain.tools import tool

@tool
def get_ethusdt_avg_price() -> str:
    """Получить среднюю цену ETHUSDT (avgPrice за последние 5 минут) из Binance."""
    url = "https://api.binance.com/api/v3/avgPrice?symbol=ETHUSDT"
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    data = r.json()
    return data["price"]


/home/emkex/life/capital/areas/code/ai_engineer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/emkex/life/capital/areas/code/ai_engineer/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
import os

from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    max_tokens=256,
)

agent = create_agent(
    model=llm,
    tools=[get_ethusdt_avg_price],
    system_prompt=(
        "Ты помощник. Если пользователь спрашивает цену ETHUSDT — "
        "используй инструмент get_ethusdt_avg_price. "
        "Ответь одной короткой фразой с ценой."
    ),
)


In [3]:
from langchain.messages import HumanMessage


result = agent.invoke({
    "messages": [
        HumanMessage(content="Сколько сейчас ETHUSDT?"),
    ]
})

print(result["messages"][-1].content)

ETHUSDT сейчас стоит **$1621.60**.


In [4]:
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Сколько сейчас стоит ETHUSDT?"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Сколько сейчас стоит ETHUSDT?
================================== Ai Message ==================================

[{'id': 'toolu_01G7BKWkhpwizuvvxv46rak9', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'get_ethusdt_avg_price', 'type': 'tool_use'}]
Tool Calls:
  get_ethusdt_avg_price (toolu_01G7BKWkhpwizuvvxv46rak9)
 Call ID: toolu_01G7BKWkhpwizuvvxv46rak9
  Args:
================================= Tool Message =================================
Name: get_ethusdt_avg_price

1621.60429780
================================== Ai Message ==================================

ETHUSDT сейчас стоит **$1621.60**.


In [5]:
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Как у тебя дела?"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Как у тебя дела?
================================== Ai Message ==================================

Спасибо за вопрос! У меня всё хорошо. Я готов помочь тебе. Если тебя интересует цена ETHUSDT или ещё что-то — с удовольствием помогу! 😊
